# Carta Azul: analysis and business review

This read-only notebook explains what the final Gold outputs enable. Run it against the authorized workspace after the pipeline has succeeded and is idle. All displayed data is aggregated; no employee names, birth dates, payroll identifiers or ticket-level records are displayed. Outputs are cleared in Git and remain private when executed in Databricks.

Accepted baseline (2026-10-27): 2,142,846 tickets across 115 days and 127 transacting locations; 424 distinct tickets flagged among 41,475 eligible employee-labeled candidates (1.0223%). The rules produced 144 recorded-tip cases and 280 discount-share cases. These are review signals, not fraud labels or estimates of misuse prevalence.

Use this notebook with `ANALYSIS_AND_PRESENTATION.md`. The code reads the current tables; the Markdown baseline describes the accepted snapshot and must not be confused with newly executed output. No source, transformation, table configuration or orchestration changes are made.


In [ ]:
import re
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "applebees"
assert all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in (CATALOG, SCHEMA))
def analysis_table(name):
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"

snapshot = spark.table(analysis_table("gold_pipeline_quality")).first().asDict()
assert snapshot["snapshot_quality_status"] != "STRUCTURAL_FAILURE"
assert snapshot["error_issue_row_count"] == 0
assert snapshot["rule_version"] == "v1_exploratory"


## 1. Establish the population before reading the cases

Monetary eligibility, ordinary-comparison eligibility and employee-labeled candidates are different populations. The flag-rate denominator is the eligible employee-labeled candidate population, not all transactions. Reconciliation exceptions use the evaluable settlement population. Missing source context can make a ticket unsuitable for comparison while its original monetary data remains available.


In [ ]:
population_summary = spark.sql(f"""
    SELECT canonical_ticket_count AS all_tickets,
           transaction_location_count AS transacting_locations,
           monetary_analysis_eligible_ticket_count AS monetary_eligible_tickets,
           ordinary_comparison_eligible_ticket_count AS ordinary_eligible_tickets,
           employee_labeled_ticket_count AS employee_labeled_tickets,
           exploratory_employee_ticket_candidate_count AS employee_labeled_candidates,
           exploratory_review_ticket_count AS flagged_distinct_tickets,
           ROUND(100.0 * exploratory_candidate_flag_rate, 4) AS candidate_flag_percent,
           reconciliation_evaluable_count,
           reconciliation_exception_count,
           ROUND(100.0 * empirical_reconciliation_pass_rate, 6) AS settlement_pass_percent,
           snapshot_quality_status
    FROM {analysis_table('gold_pipeline_quality')}
""")
display(population_summary)


## 2. Explain the review queue

The recorded-tip rule requires both the tip amount and tip/gross ratio to exceed a historical peer percentile. Its peers have no employee-discount label and match channel, weekday, card context and location where sufficient support exists. The discount rule compares the employee discount/gross ratio within an exact normalized employee-label set. Small local cohorts fall back to a broader location pool; support must still be at least 50. Percentile 0.99 and support 50 are exploratory choices.

A ticket can produce multiple reasons; always calculate distinct tickets as well as case rows. In the accepted snapshot, the two exploratory reason sets happened not to overlap. That observation is not a general guarantee. Quality cases have a different interpretation and are counted separately.


In [ ]:
review_summary = spark.sql(f"""
    SELECT case_category, reason_code, COUNT(*) AS case_rows,
           COUNT(DISTINCT ticket_key) AS distinct_tickets
    FROM {analysis_table('gold_review_cases')}
    GROUP BY case_category, reason_code
    ORDER BY case_category, reason_code
""")
display(review_summary)

exploratory_summary = review_summary.where("case_category = 'EXPLORATORY_COMPARISON'")
chart_rows = exploratory_summary.orderBy("reason_code").collect()
# Databricks also supports charting the displayed aggregate result directly.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.5))
labels = [r.reason_code.replace("EMPLOYEE_LABEL_HIGH_", "").replace("_", " ").title() for r in chart_rows]
values = [r.distinct_tickets for r in chart_rows]
ax.barh(labels, values, color=["#177e89", "#3659a2"][:len(values)])
for i, value in enumerate(values):
    ax.text(value, i, f" {value:,}", va="center")
ax.set_xlabel("Distinct tickets flagged for exploratory review")
ax.set_title("Historical review queue by reason")
ax.set_xlim(0, max(values, default=0) * 1.2 + 1)
fig.tight_layout()
plt.show()
plt.close(fig)


## 3. Put monthly volumes in context

Count eligible candidates and flags in the same month, then divide. This is a descriptive breakdown of rules fitted to the complete historical dataset, not a prospective evaluation or proof of a trend. April and August are partial months, so raw monthly totals are not directly comparable as rates of daily activity. Recalculate when the rule version or input snapshot changes.


In [ ]:
monthly_context = spark.sql(f"""
    WITH candidates AS (
        SELECT DATE_TRUNC('MONTH', business_date)::DATE AS business_month,
               COUNT(*) AS eligible_employee_candidates
        FROM {analysis_table('gold_check_features')}
        WHERE ordinary_comparison_eligible AND has_employee_discount
        GROUP BY DATE_TRUNC('MONTH', business_date)::DATE
    ), flagged AS (
        SELECT DATE_TRUNC('MONTH', business_date)::DATE AS business_month,
               COUNT(DISTINCT ticket_key) AS flagged_tickets
        FROM {analysis_table('gold_review_cases')}
        WHERE case_category = 'EXPLORATORY_COMPARISON'
        GROUP BY DATE_TRUNC('MONTH', business_date)::DATE
    )
    SELECT c.business_month, c.eligible_employee_candidates,
           COALESCE(f.flagged_tickets, 0) AS flagged_tickets,
           ROUND(100.0 * TRY_DIVIDE(COALESCE(f.flagged_tickets, 0),
                                   c.eligible_employee_candidates), 4) AS candidate_flag_percent
    FROM candidates c LEFT JOIN flagged f ON c.business_month = f.business_month
    ORDER BY c.business_month
""")
display(monthly_context)


## 4. Review location context without a raw-count leaderboard

A location with more transactions can produce more flags without a higher flag rate. The table below summarizes locations by candidate support and flags; it omits location identifiers. To investigate a specific location, an authorized analyst can drill into the private tables using the queries in the presentation guide. Compare cohort support and channel mix before prioritizing a site. These rules describe transactions; they do not identify a beneficiary.


In [ ]:
location_context = spark.sql(f"""
    WITH candidates AS (
        SELECT location_id, COUNT(*) AS candidate_tickets
        FROM {analysis_table('gold_check_features')}
        WHERE ordinary_comparison_eligible AND has_employee_discount
        GROUP BY location_id
    ), flagged AS (
        SELECT location_id, COUNT(DISTINCT ticket_key) AS flagged_tickets
        FROM {analysis_table('gold_review_cases')}
        WHERE case_category = 'EXPLORATORY_COMPARISON'
        GROUP BY location_id
    ), location_rates AS (
        SELECT c.location_id, c.candidate_tickets,
               COALESCE(f.flagged_tickets, 0) AS flagged_tickets,
               TRY_DIVIDE(COALESCE(f.flagged_tickets, 0), c.candidate_tickets) AS flag_rate
        FROM candidates c LEFT JOIN flagged f ON c.location_id = f.location_id
    )
    SELECT CASE WHEN candidate_tickets < 50 THEN 'Fewer than 50 candidates'
                ELSE 'At least 50 candidates' END AS location_candidate_support,
           COUNT(*) AS location_count,
           SUM(candidate_tickets) AS candidate_tickets,
           SUM(flagged_tickets) AS flagged_tickets,
           COUNT_IF(flagged_tickets > 0) AS locations_with_flags,
           ROUND(100.0 * TRY_DIVIDE(SUM(flagged_tickets), SUM(candidate_tickets)), 4)
               AS pooled_candidate_flag_percent
    FROM location_rates
    GROUP BY CASE WHEN candidate_tickets < 50 THEN 'Fewer than 50 candidates'
                  ELSE 'At least 50 candidates' END
    ORDER BY location_candidate_support
""")
display(location_context)


## 5. Use daily discount rates with the correct denominator

`gold_discount_daily` has one row per date/location/channel/discount label and identifier. Its population denominators repeat across discount labels. Filter to one label when using that table's frequency column. For a monthly employee-ticket frequency, calculate one row per ticket from `gold_check_features` instead, as below. Do not add label frequencies or repeated denominators across labels.


In [ ]:
monthly_employee_frequency = spark.sql(f"""
    SELECT DATE_TRUNC('MONTH', business_date)::DATE AS business_month,
           COUNT(*) AS monetary_eligible_tickets,
           COUNT_IF(discount_classification_complete) AS classifiable_tickets,
           COUNT_IF(discount_classification_complete AND has_employee_discount)
               AS employee_labeled_tickets,
           ROUND(100.0 * TRY_DIVIDE(
               COUNT_IF(discount_classification_complete AND has_employee_discount),
               COUNT_IF(discount_classification_complete)), 4)
               AS employee_labeled_percent_of_classifiable_tickets
    FROM {analysis_table('gold_check_features')}
    WHERE monetary_analysis_eligible
    GROUP BY DATE_TRUNC('MONTH', business_date)::DATE
    ORDER BY business_month
""")
display(monthly_employee_frequency)
print("GOLD_ANALYSIS_PASS")


## 6. Business interpretation and next steps

The pipeline turns nested JSON into a repeatable, explainable review queue with a defined denominator. Review the quality exceptions before interpreting behavior, inspect private transaction context for a flagged case, confirm the meaning of discounts with the business, and record the review outcome separately from the exploratory flag. Do not infer savings, causality, benefit-policy breaches or employee wrongdoing from these outputs. Cash tips and official discount policies are unavailable.

A sensitivity exercise can compare alternative percentiles and minimum supports later, without presenting the current settings as official policy. Such a study and a measured clustering experiment are future analytical/performance work, not results of this notebook.
